# Predicting Familial vs. Sporadic NF1 Cases

**NJIT DS675 Machine Learning** | Linear Regression versus Linear SVM

This notebook compares familial and sporadic NF1 cases using 18 predictors and stratified five fold cross-validation with balanced accuracy. It also compares results after removing three age-related predictors.

**Educational research only. Not for clinical diagnosis.**

## Dataset and setup

1. Obtain `dataset-uci.xlsx` from the [UCI NF1 dataset](https://archive.ics.uci.edu/dataset/1162/neurofibromatosis+type+1+clinical+symptoms+of+familial+and+sporadic+cases).
2. Put it at `data/dataset-uci.xlsx` in this repository.
3. Install the packages in `requirements.txt`.
4. Run the notebook from top to bottom using Python/Jupyter.

The original saved notebook's working data had **296 records** and **18 predictors**, while the current UCI listing indicates 331 instances. Confirm the dataset version before comparing results. This adapted notebook removes the original Google Drive mounting and ZIP extraction steps. Saved execution outputs have been removed because the notebook has not yet been rerun against the source spreadsheet.

Source and attribution: Sharafi et al. (2025), UCI Machine Learning Repository; [associated article](https://doi.org/10.5505/TurkHijyen.2025.06337).


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

# Works from either the repository root or the notebooks/ directory.
repo_root = Path.cwd()
if not (repo_root / "data").exists() and (repo_root.parent / "data").exists():
    repo_root = repo_root.parent

data_path = repo_root / "data" / "dataset-uci.xlsx"
if not data_path.is_file():
    raise FileNotFoundError(
        "Dataset missing. Download dataset-uci.xlsx and place it at data/dataset-uci.xlsx. "
        "See data/README.md."
    )
df = pd.read_excel(data_path)
print("Loaded data shape:", df.shape)
df.head()


In [ ]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
for column in df.columns:
    print(column)

In [ ]:
df.info()

In [ ]:
excel_file = pd.ExcelFile(data_path)

print(excel_file.sheet_names)

In [ ]:
class_labels = pd.read_excel(
    data_path,
    sheet_name="Class Labels"
)

pd.set_option("display.max_colwidth", None)
print(class_labels.to_string(index=False))

In [ ]:
variables = pd.read_excel(
    data_path,
    sheet_name="Variables table"
)

print(variables.to_string(index=False))

In [ ]:
X = df.drop(columns=["Unnamed: 0", "Case Type"])
y = df["Case Type"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
print("Number of cases:")
print(y.value_counts())

print("\nPercentage of cases:")
print(y.value_counts(normalize=True) * 100)

In [ ]:
print(X.isnull().sum())

In [ ]:
X[["Age of Mother", "Age of Father", "Age at First Diagnosis"]].describe()

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.base import clone

In [ ]:
# Create the Linear Regression pipeline
linear_reg_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

# Set up 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store the balanced accuracy score from each fold
lr_scores = []

# Run the 5 folds
for train_index, test_index in cv.split(X, y):

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    # Create a fresh copy of the pipeline for this fold
    model = clone(linear_reg_pipeline)

    # Train the model
    model.fit(X_train, y_train)

    # Linear Regression gives continuous predictions
    predictions = model.predict(X_test)

    # Convert them to class predictions
    # Below 0.5 = Sporadic (0)
    # 0.5 or above = Familial (1)
    predicted_classes = (predictions >= 0.5).astype(int)

    # Calculate balanced accuracy
    score = balanced_accuracy_score(y_test, predicted_classes)
    lr_scores.append(score)

# Display results
print("Balanced accuracy for each fold:", lr_scores)
print("Mean balanced accuracy:", np.mean(lr_scores))
print("Standard deviation:", np.std(lr_scores))

In [ ]:
# C values to test
c_values = [100, 10, 1, 0.1, 0.01]

# Store the results
svm_results = {}

# Test each C value
for c in c_values:

    svm_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearSVC(C=c, max_iter=10000))
    ])

    fold_scores = []

    # Use the same 5-fold cross-validation
    for train_index, test_index in cv.split(X, y):

        X_train = X.iloc[train_index]
        X_test = X.iloc[test_index]
        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]

        model = clone(svm_pipeline)

        # Train the SVM
        model.fit(X_train, y_train)

        # Predict Sporadic (0) or Familial (1)
        predicted_classes = model.predict(X_test)

        # Calculate balanced accuracy
        score = balanced_accuracy_score(y_test, predicted_classes)
        fold_scores.append(score)

    # Save the results for this C value
    svm_results[c] = {
        "fold_scores": fold_scores,
        "mean": np.mean(fold_scores),
        "std": np.std(fold_scores)
    }

    print(f"C = {c}")
    print("Fold scores:", fold_scores)
    print("Mean balanced accuracy:", np.mean(fold_scores))
    print("Standard deviation:", np.std(fold_scores))
    print()

In [ ]:
# Diagnostic check: compare Linear Regression and SVM predictions

for fold_number, (train_index, test_index) in enumerate(cv.split(X, y), start=1):

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    y_train = y.iloc[train_index]

    # Linear Regression
    lr_model = clone(linear_reg_pipeline)
    lr_model.fit(X_train, y_train)

    lr_predictions = lr_model.predict(X_test)
    lr_classes = (lr_predictions >= 0.5).astype(int)

    # Linear SVM with C = 1
    svm_check = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearSVC(C=1, max_iter=10000))
    ])

    svm_check.fit(X_train, y_train)
    svm_classes = svm_check.predict(X_test)

    # Count how many predictions are different
    differences = np.sum(lr_classes != svm_classes)

    print(
        f"Fold {fold_number}: "
        f"{differences} different predictions out of {len(X_test)}"
    )

In [ ]:
# Create a list for the results table
results_data = []

# Add Linear Regression results
results_data.append({
    "Model": "Linear Regression",
    "C": "N/A",
    "Mean Balanced Accuracy": np.mean(lr_scores),
    "Standard Deviation": np.std(lr_scores)
})

# Add Linear SVM results
for c in c_values:
    results_data.append({
        "Model": "Linear SVM",
        "C": c,
        "Mean Balanced Accuracy": svm_results[c]["mean"],
        "Standard Deviation": svm_results[c]["std"]
    })

# Convert to a DataFrame
results_df = pd.DataFrame(results_data)

# Display percentages for easier interpretation
results_df["Mean Balanced Accuracy (%)"] = (
    results_df["Mean Balanced Accuracy"] * 100
).round(2)

results_df["Standard Deviation (%)"] = (
    results_df["Standard Deviation"] * 100
).round(2)

# Show the clean table
results_df[
    ["Model", "C", "Mean Balanced Accuracy (%)", "Standard Deviation (%)"]
]

In [ ]:
import matplotlib.pyplot as plt

# Get SVM mean accuracies in the same order as c_values
svm_means = [svm_results[c]["mean"] * 100 for c in c_values]

# Create the graph
plt.figure(figsize=(8, 5))

plt.plot(
    [str(c) for c in c_values],
    svm_means,
    marker="o"
)

# Add Linear Regression performance as a reference line
plt.axhline(
    y=np.mean(lr_scores) * 100,
    linestyle="--",
    label="Linear Regression"
)

plt.xlabel("C Value")
plt.ylabel("Mean Balanced Accuracy (%)")
plt.title("Linear SVM Performance Across C Values")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(
    os.path.join(project_folder, "svm_c_performance.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

# Representative Linear SVM model
svm_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearSVC(C=1, max_iter=10000))
])

# Generate predictions for every patient using 5-fold cross-validation
svm_cv_predictions = cross_val_predict(
    svm_final,
    X,
    y,
    cv=cv
)

# Confusion matrix
cm = confusion_matrix(y, svm_cv_predictions)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y,
        svm_cv_predictions,
        target_names=["Sporadic", "Familial"]
    )
)

In [ ]:
# Train the representative SVM on the full dataset
svm_final.fit(X, y)

# Get the SVM coefficients
coefficients = svm_final.named_steps["model"].coef_[0]

# Create a table matching each feature to its coefficient
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coefficients
})

# Add absolute coefficient size
feature_importance["Absolute Coefficient"] = (
    feature_importance["Coefficient"].abs()
)

# Sort from strongest to weakest
feature_importance = feature_importance.sort_values(
    by="Absolute Coefficient",
    ascending=False
)

feature_importance

In [ ]:
# Sort features by coefficient for the graph
plot_data = feature_importance.sort_values(
    by="Coefficient",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    plot_data["Feature"],
    plot_data["Coefficient"]
)

# Add a vertical line at zero
plt.axvline(x=0, linewidth=1)

plt.xlabel("SVM Coefficient")
plt.ylabel("Feature")
plt.title("Feature Coefficients in Linear SVM (C = 1)")

plt.tight_layout()
plt.savefig(
    os.path.join(project_folder, "svm_c_performance.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# Compare average feature values between Sporadic and Familial cases

comparison_data = df.drop(columns=["Unnamed: 0"]).groupby("Case Type").mean().T

comparison_data.columns = ["Sporadic Mean", "Familial Mean"]

comparison_data["Absolute Difference"] = (
    comparison_data["Familial Mean"] -
    comparison_data["Sporadic Mean"]
).abs()

comparison_data = comparison_data.sort_values(
    by="Absolute Difference",
    ascending=False
)

comparison_data

In [ ]:
# Exclude the three continuous age variables
binary_comparison = comparison_data.drop(
    index=["Age of Mother", "Age of Father", "Age at First Diagnosis"]
).copy()

# Convert proportions to percentages
binary_plot = binary_comparison[
    ["Sporadic Mean", "Familial Mean"]
] * 100

# Sort by the size of the difference between the groups
binary_plot["Difference"] = (
    binary_plot["Familial Mean"] -
    binary_plot["Sporadic Mean"]
).abs()

binary_plot = binary_plot.sort_values(
    by="Difference",
    ascending=True
)

# Create graph
ax = binary_plot[
    ["Sporadic Mean", "Familial Mean"]
].plot(
    kind="barh",
    figsize=(10, 8)
)

plt.xlabel("Patients With Feature (%)")
plt.ylabel("Clinical Feature")
plt.title("Clinical Feature Prevalence: Sporadic vs Familial NF1")
plt.legend(["Sporadic", "Familial"])

plt.tight_layout()

# Save permanently to Google Drive
plt.savefig(
    os.path.join(project_folder, "sporadic_vs_familial_features.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Create a second feature set without the three age variables

age_features = [
    "Age of Mother",
    "Age of Father",
    "Age at First Diagnosis"
]

X_no_age = X.drop(columns=age_features)

print("Original number of features:", X.shape[1])
print("Number of features without age variables:", X_no_age.shape[1])

print("\nRemaining features:")
for feature in X_no_age.columns:
    print(feature)

In [ ]:
# Linear Regression without the three age features

linear_reg_no_age = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

lr_no_age_scores = []

for train_index, test_index in cv.split(X_no_age, y):

    X_train = X_no_age.iloc[train_index]
    X_test = X_no_age.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    model = clone(linear_reg_no_age)

    # Train the model
    model.fit(X_train, y_train)

    # Get continuous predictions
    predictions = model.predict(X_test)

    # Convert predictions into Sporadic (0) or Familial (1)
    predicted_classes = (predictions >= 0.5).astype(int)

    # Calculate balanced accuracy
    score = balanced_accuracy_score(y_test, predicted_classes)
    lr_no_age_scores.append(score)

print("Balanced accuracy for each fold:", lr_no_age_scores)
print("Mean balanced accuracy:", np.mean(lr_no_age_scores))
print("Standard deviation:", np.std(lr_no_age_scores))

In [ ]:
# Linear SVM without the three age features

c_values = [100, 10, 1, 0.1, 0.01]

svm_no_age_results = {}

for c in c_values:

    svm_pipeline_no_age = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearSVC(C=c, max_iter=10000))
    ])

    fold_scores = []

    for train_index, test_index in cv.split(X_no_age, y):

        X_train = X_no_age.iloc[train_index]
        X_test = X_no_age.iloc[test_index]
        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]

        model = clone(svm_pipeline_no_age)
        model.fit(X_train, y_train)

        predicted_classes = model.predict(X_test)

        score = balanced_accuracy_score(
            y_test,
            predicted_classes
        )

        fold_scores.append(score)

    svm_no_age_results[c] = {
        "fold_scores": fold_scores,
        "mean": np.mean(fold_scores),
        "std": np.std(fold_scores)
    }

for c, results in svm_no_age_results.items():
    print(f"C = {c}")
    print("Fold scores:", results["fold_scores"])
    print("Mean balanced accuracy:", results["mean"])
    print("Standard deviation:", results["std"])
    print()

In [ ]:
# Compare model performance with and without age features

comparison_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Linear SVM",
        "Linear SVM",
        "Linear SVM",
        "Linear SVM",
        "Linear SVM"
    ],
    "C": ["N/A", 100, 10, 1, 0.1, 0.01],
    "With Age (%)": [
        np.mean(lr_scores) * 100,
        svm_results[100]["mean"] * 100,
        svm_results[10]["mean"] * 100,
        svm_results[1]["mean"] * 100,
        svm_results[0.1]["mean"] * 100,
        svm_results[0.01]["mean"] * 100
    ],
    "Without Age (%)": [
        np.mean(lr_no_age_scores) * 100,
        svm_no_age_results[100]["mean"] * 100,
        svm_no_age_results[10]["mean"] * 100,
        svm_no_age_results[1]["mean"] * 100,
        svm_no_age_results[0.1]["mean"] * 100,
        svm_no_age_results[0.01]["mean"] * 100
    ]
})

comparison_results["Change (%)"] = (
    comparison_results["Without Age (%)"]
    - comparison_results["With Age (%)"]
)

comparison_results.round(2)

In [ ]:
import matplotlib.pyplot as plt

labels = [
    "Linear Regression",
    "SVM C=100",
    "SVM C=10",
    "SVM C=1",
    "SVM C=0.1",
    "SVM C=0.01"
]

with_age = comparison_results["With Age (%)"]
without_age = comparison_results["Without Age (%)"]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(11, 6))

plt.bar(x - width/2, with_age, width, label="With Age Features")
plt.bar(x + width/2, without_age, width, label="Without Age Features")

plt.ylabel("Mean Balanced Accuracy (%)")
plt.xlabel("Model Configuration")
plt.title("Effect of Excluding Age Features on Model Performance")

plt.xticks(x, labels, rotation=30, ha="right")
plt.legend()

# Make the small differences easier to see
plt.ylim(45, 56)

plt.tight_layout()

# Save permanently to Google Drive
comparison_graph_path = os.path.join(
    project_folder,
    "age_feature_comparison.png"
)

plt.savefig(comparison_graph_path, dpi=300, bbox_inches="tight")
plt.show()

print("Graph saved to:", comparison_graph_path)